In [6]:
# Required Libraries

import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import pandas as pd
import requests
import time

#### Acquire `.xml` files from [datacentermap.com](https://www.datacentermap.com/sitemap/)

In [7]:
SITEMAP_INDEX = "https://www.datacentermap.com/sitemap.xml"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
}

def fetch_xml(url, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)

            if r.status_code == 429:
                wait = 2 ** attempt  # exponential backoff
                print(f"429 received. Sleeping {wait}s...")
                time.sleep(wait)
                continue

            r.raise_for_status()
            return r.text

        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"Error: {e}. Retrying in {wait}s...")
            time.sleep(wait)

    raise Exception("Failed after retries")

def parse_sitemap_index(xml_text):
    soup = BeautifulSoup(xml_text, "xml")
    return [loc.text.strip() for loc in soup.find_all("loc")]

def parse_urlset(xml_text):
    soup = BeautifulSoup(xml_text, "xml")
    return [loc.text.strip() for loc in soup.find_all("loc")]

In [8]:
# 1. download the sitemap index
index_xml = fetch_xml(SITEMAP_INDEX)

In [9]:
# 2. extract child sitemap URLs
child_sitemaps = parse_sitemap_index(index_xml)
print(f"Found {len(child_sitemaps)} child sitemap files")
print(child_sitemaps[:10])

Found 30 child sitemap files
['https://www.datacentermap.com/sitemap/site.xml', 'https://www.datacentermap.com/sitemap/blog.xml', 'https://www.datacentermap.com/sitemap/ixps.xml', 'https://www.datacentermap.com/sitemap/dcs_1.xml', 'https://www.datacentermap.com/sitemap/dcs_2.xml', 'https://www.datacentermap.com/sitemap/dcs_3.xml', 'https://www.datacentermap.com/sitemap/dcs_4.xml', 'https://www.datacentermap.com/sitemap/dcs_5.xml', 'https://www.datacentermap.com/sitemap/dcs_6.xml', 'https://www.datacentermap.com/sitemap/dcs_7.xml']


In [10]:
# 3. download each child sitemap and extract URLs
all_urls = []

for i, sitemap_url in enumerate(child_sitemaps, start=1):
    try:
        xml_text = fetch_xml(sitemap_url)
        urls = parse_urlset(xml_text)
        all_urls.extend(urls)
        print(f"[{i}/{len(child_sitemaps)}] {sitemap_url} -> {len(urls)} URLs")
        time.sleep(0.5)  # be polite
    except Exception as e:
        print(f"Failed on {sitemap_url}: {e}")

[1/30] https://www.datacentermap.com/sitemap/site.xml -> 24 URLs
[2/30] https://www.datacentermap.com/sitemap/blog.xml -> 38 URLs
[3/30] https://www.datacentermap.com/sitemap/ixps.xml -> 971 URLs
[4/30] https://www.datacentermap.com/sitemap/dcs_1.xml -> 1000 URLs
[5/30] https://www.datacentermap.com/sitemap/dcs_2.xml -> 1000 URLs
[6/30] https://www.datacentermap.com/sitemap/dcs_3.xml -> 1000 URLs
[7/30] https://www.datacentermap.com/sitemap/dcs_4.xml -> 1000 URLs
[8/30] https://www.datacentermap.com/sitemap/dcs_5.xml -> 1000 URLs
[9/30] https://www.datacentermap.com/sitemap/dcs_6.xml -> 1000 URLs
[10/30] https://www.datacentermap.com/sitemap/dcs_7.xml -> 1000 URLs
[11/30] https://www.datacentermap.com/sitemap/dcs_8.xml -> 1000 URLs
[12/30] https://www.datacentermap.com/sitemap/dcs_9.xml -> 1000 URLs
[13/30] https://www.datacentermap.com/sitemap/dcs_10.xml -> 1000 URLs
[14/30] https://www.datacentermap.com/sitemap/dcs_11.xml -> 1000 URLs
[15/30] https://www.datacentermap.com/sitemap/dcs

In [11]:
# 4. de-duplicate
all_urls = sorted(set(all_urls))
print(f"Total unique URLs: {len(all_urls)}")

Total unique URLs: 25734


In [12]:
# 5. save raw results
pd.DataFrame({"url": all_urls}).to_csv("datacentermap_all_urls.csv", index=False)
print("Saved datacentermap_all_urls.csv")

Saved datacentermap_all_urls.csv


#### Parse the sitemap XMLs to extract all datacenter URLs

In [18]:
# Required Libraries

import regex as re                  # Extended regular expressions for robust pattern matching
import xml.etree.ElementTree as ET  # For parsing XML sitemaps
import os                           # For file system operations
import csv                          # For writing CSV outputs

In [19]:
# Define input and output paths
input_folder = "../input/DCMap"                      
output_file = "../output/01datacenter_urls.csv" 

In [ ]:
# Define helper function to extract <loc> tags from XML sitemap files
def extract_urls_from_file(file_path):
    """
    Parses an XML sitemap and extracts all <loc> tags containing data center URLs.
    Returns a list of URLs or an empty list if the file is unreadable.
    """
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        namespace = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
        
        # Extract text content of each <loc> tag
        urls = [url.text.strip() for url in root.findall(".//ns:loc", namespace)]
        return urls
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return []